In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time
import pandas as pd

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Compute

In [ ]:
def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom

In [ ]:
## load input data
forced, anom = load_consolidated_wide()

## compute
ohc = src.utils.reconstruct_wrapper(
    anom[["T","T_comp"]],
    lambda x : x.integrate("z_t"),
)

## save to file

In [ ]:
print(list((DATA_FP / "cesm").glob("*")))

In [ ]:
save_fp = pathlib.Path(DATA_FP, "cesm", "ohc_est.nc")
ohc.sel(longitude=slice(120,285)).to_netcdf(save_fp)

## Compare to h

In [ ]:
hw_ohc = ohc.sel(longitude=slice(120,210)).mean("longitude")["T"]
he_ohc = ohc.sel(longitude=slice(210,270)).mean("longitude")["T"]

In [ ]:
hw = src.utils.reconstruct_wrapper(
    anom[["ssh","ssh_comp"]], 
    src.utils.get_h